# Query Text Data (v2)

Runs the `itk_query_text_data` package step by step over a sample of news articles: chunk the documents, build a concept graph, embed the chunks, search for relevant chunks, and write a cited report.

See the [package README](../packages/itk-query-text-data/README.md) for how v2 differs from v1 (`query_text_data.ipynb`).

Install with `uv sync --extra dev` from the repository root, then set the AI connection as environment variables before starting Jupyter:

- **OpenAI:** `OPENAI_API_KEY`, and optionally `OPENAI_API_MODEL`.
- **Azure OpenAI:** `OPENAI_TYPE="Azure OpenAI"`, `AZURE_OPENAI_ENDPOINT`, `OPENAI_API_MODEL` (deployment name), `OPENAI_EMBEDDING_MODEL`, and `AZURE_AUTH_TYPE`, which is either `"Azure Key"` (with `OPENAI_API_KEY`) or `"Managed Identity"` (uses `az login`).

In [1]:
import os

os.environ["MKL_THREADING_LAYER"] = "GNU"  # Avoids threadpoolctl error in Linux and MacOS

from IPython.display import Markdown
from IPython.display import display as show
from itk_core.openai_configuration import OpenAIConfiguration
from itk_core.openai_embedder import OpenAIEmbedder

import itk_query_text_data as qtd

ai_configuration = OpenAIConfiguration()  # Read from the environment variables above
llm = qtd.OpenAILLM(ai_configuration)
embedder = qtd.ItkEmbedder(OpenAIEmbedder(ai_configuration))
print(f"{ai_configuration.api_type}: {ai_configuration.model}, embeddings: {ai_configuration.embedding_model}")

Azure OpenAI: gpt-5.4, embeddings: text-embedding-3-small


## 1. Load and chunk documents

`load_documents(paths, chunk_size)` reads and chunks in one step. Here we read first and keep a sample, so the notebook runs quickly. Each CSV row becomes its own document.

In [2]:
input_path = "../example_outputs/query_text_data/news_articles/news_articles_texts.csv"
max_documents = 200  # Increase, or use every row, for a fuller analysis

documents = qtd.read_documents([input_path])[:max_documents]
corpus = qtd.chunk_documents(documents, chunk_size=500)
print(f"{len(documents)} documents -> {len(corpus)} chunks")
print(corpus[1].text[:300])

200 documents -> 200 chunks
mock_text: **Aftershocks in Indonesia: Local and International Aid Continues**

**Bali, Indonesia – July 16, 2023** – In the wake of the devastating earthquake that struck the Indonesian archipelago, aftershocks continue to rattle the region, with significant tremors felt from Bali to Surabaya. Th


## 2. Build the concept graph

Extracts noun-phrase concepts from each chunk, links concepts that co-occur, and groups them into hierarchical Leiden communities. This step is local and needs no AI calls.

In [3]:
graph = qtd.build_concept_graph(corpus)
print(f"{graph.graph.number_of_nodes()} concepts, {graph.graph.number_of_edges()} links, "
      f"{len(graph.communities)} communities over {graph.max_level + 1} levels")
print("Top concepts:", ", ".join(graph.top_concepts(15)))

165 concepts, 264 links, 19 communities over 1 levels
Top concepts: essential supplies, relief efforts, dj alex groove, harmony park, tech innovators, global relief network, temporary shelters, capitol city, electronic music scene, food stalls, healthcare reform, red cross, tech industry, annual event, broad spectrum


## 3. Embed the chunks

Embeddings are cached on disk by `itk_core`, so rerunning is fast.

In [4]:
embeddings = await qtd.embed_chunks(corpus, embedder)
print(f"Embedded {len(embeddings)} chunks")

Embedded 200 chunks


## 4. Search

`rewrite_query` anchors the question to the corpus's main concepts. `search` then tests chunks for relevance community by community, within `relevance_test_budget`, and groups the relevant ones into themes. `on_event` reports progress as it goes.

In [5]:
query = "What are the main humanitarian impacts of natural disasters, and how are communities responding?"

expanded_query = await qtd.rewrite_query(query, graph, llm)
print("Expanded query:", expanded_query)

config = qtd.SearchConfig(relevance_test_budget=30, analysis_update_interval=10)


def on_event(event: qtd.Event) -> None:
    if isinstance(event, qtd.ChunksTested):
        print(f"Tested {len(event.tests)} chunks, {len(event.relevant_chunk_ids)} relevant")
    elif isinstance(event, qtd.AnalysisUpdated):
        print(f"Analysis updated: {len(event.structure.themes)} themes")


result = await qtd.search(
    query, corpus, graph, embeddings,
    llm=llm, embedder=embedder, expanded_query=expanded_query,
    config=config, on_event=on_event,
)
show(Markdown(result.summary))

Expanded query: What are the main humanitarian impacts of natural disasters, especially in hardest-hit areas, and how are communities, local authorities, and organizations like the Red Cross and global relief networks responding through immediate relief, essential supplies, temporary shelters, clean water, medical supplies, relief operations, and long-term recovery efforts?


Tested 0 chunks, 0 relevant


Tested 5 chunks, 5 relevant


Tested 10 chunks, 10 relevant


Analysis updated: 3 themes


...

Tested 22 chunks, 11 relevant


Tested 27 chunks, 11 relevant


Tested 30 chunks, 11 relevant


Analysis updated: 3 themes


**Relevant chunks / tested chunks: 11/30** (topic 1: 5/5; topic 1.1: 5/5; topic 1.19: 1/3; <span style='color: red'>topic 1.17: 0/2</span>; <span style='color: red'>topic 1.2: 0/5</span>; <span style='color: red'>topic 1.15: 0/2</span>; <span style='color: red'>topic 1.10: 0/5</span>; <span style='color: red'>topic 1.5: 0/3</span>)

In [6]:
for theme, chunk_ids in result.themes.items():
    print(f"{theme}: chunks {list(chunk_ids)}")

Humanitarian impacts on affected populations: chunks [1, 2, 121, 122, 123, 3, 4]
Damage and operational barriers caused by the disasters: chunks [1, 2, 124, 122]
Community and institutional responses for relief and recovery: chunks [1, 2, 3, 4, 121, 122, 123]


## 5. AI commentary (optional)

A short commentary on the themes found. `on_token` receives the text so far, so the display updates as it streams.

In [7]:
commentary_display = show(Markdown(""), display_id=True)
commentary = await qtd.generate_commentary(
    query, dict(result.themes), result.analysis_markdown, corpus, llm,
    on_token=lambda text: commentary_display.update(Markdown(text)),
)
commentary_display.update(Markdown(commentary))

The themes show a clear progression from immediate human suffering, to the practical barriers created by the disasters, to the layered response by communities and institutions. [source: 1, 2, 3, 121, 122, 124] Together, the sources portray disasters not just as physical events, but as crises that disrupt health, shelter, education, livelihoods, and the systems needed to deliver help. [source: 1, 2, 3, 4, 121, 122, 123, 124]

The main humanitarian impacts begin with urgent survival needs among affected populations. [source: 2, 121, 122, 123] Reports from Jakarta, Medan, Lombok, Bali, and Denpasar describe communities needing food, clean water, temporary shelter, and medical supplies, with displaced people relying on temporary shelters and emergency distributions. [source: 2, 121, 122, 123]

Health impacts are another major theme, especially where earthquakes have generated mass casualties and strained care systems. [source: 2, 3, 122] In Bali and Lombok, hospitals were described as overwhelmed, while international and local teams provided trauma care, emergency treatment, infectious-disease management, and preventive health support to reduce post-disaster illness risks. [source: 3, 122]

The humanitarian consequences also extend beyond physical injury to psychosocial and child-centered needs. [source: 4, 122] In Jakarta and Bali, responders created temporary learning centers, safe spaces, and educational support while also integrating psychological support for children and for people who had lost their homes. [source: 4, 122]

A second major part of the story is that the disasters damage the very infrastructure needed for everyday life and relief operations. [source: 1, 2, 124] Bali saw considerable infrastructure damage, Surabaya experienced disruptions to daily life and the local economy, and Medan faced access problems because damaged infrastructure made some areas difficult to reach. [source: 1, 2]

These physical impacts translated into operational barriers for humanitarian action. [source: 2, 122, 124] Aftershocks, transportation delays, inadequate infrastructure, and weak coordination created redundancies and gaps in aid delivery, especially in Surabaya and nearby Yogyakarta, leading to calls for a centralized command and more integrated planning. [source: 122, 124]

Communities are responding through a mix of local resilience and organized humanitarian coordination. [source: 1, 2, 4, 121, 122, 123] The Indonesian Red Cross, Global Relief Network, UNICEF, WHO, Save the Children, Global Aid Society, and the Emergency Response Alliance are shown working with local authorities and volunteers to distribute aid, set up shelters and clinics, and assess damage. [source: 1, 2, 3, 4, 121, 122, 123]

The response is not limited to short-term relief, but also includes recovery and preparedness. [source: 1, 3, 121, 123] Several sources emphasize rebuilding homes, schools, hospitals, healthcare infrastructure, and essential services, while also strengthening resilience and preparing communities for future seismic events. [source: 1, 3, 121]

Specific examples help illustrate this broader pattern. [source: 2, 3, 4, 124] In Medan, volunteers established temporary shelters and makeshift clinics despite difficult access, in Bali and Lombok medical teams reinforced overwhelmed hospitals, in Jakarta and Bali educators and aid groups set up temporary learning spaces, and in Surabaya officials organized workshops and coordination meetings to improve aid delivery. [source: 2, 3, 4, 124]

Overall, the main humanitarian impacts are immediate deprivation, health emergencies, trauma, displacement, and service disruption, while the main responses are coordinated relief delivery, child and health protection measures, infrastructure recovery, and efforts to improve disaster-response coordination. [source: 1, 2, 3, 4, 121, 122, 123, 124] The sources suggest that communities are not responding passively; rather, they are working with national and international partners to meet urgent needs while trying to rebuild stronger systems for the future. [source: 1, 4, 121, 124]

## 6. Build the report

`build_answer` summarizes each theme and writes the final report, with citations linked to the source chunks.

In [8]:
answer = await qtd.build_answer(result, corpus, llm)
print(f"Cites {len(answer.references)} chunks")
show(Markdown(answer.markdown))

Cites 7 chunks


## Query

*What are the main humanitarian impacts of natural disasters, and how are communities responding?*

## Expanded Query

*What are the main humanitarian impacts of natural disasters, especially in hardest-hit areas, and how are communities, local authorities, and organizations like the Red Cross and global relief networks responding through immediate relief, essential supplies, temporary shelters, clean water, medical supplies, relief operations, and long-term recovery efforts?*

## Answer

The main humanitarian impacts of natural disasters in the Indonesia reports are displacement, destruction of homes and infrastructure, disruption to daily life and local economies, and severe strain on health services. Affected communities in places including Bali, Lombok, Surabaya, Jakarta, Medan, and Denpasar were described as coping with major destruction, with some areas identified as hardest hit and daily life disrupted. [source: [1](#source-1), [2](#source-2), [3](#source-3), [122](#source-122), [123](#source-123)] Infrastructure damage made some locations difficult to reach, damaged local systems and services, and created safety concerns, while aftershocks in places such as Surabaya and Lombok continued to threaten residents and complicate operations. [source: [1](#source-1), [2](#source-2), [121](#source-121), [122](#source-122)] Many people were displaced and needed temporary shelter and urgent assistance, and local hospitals were overwhelmed by incoming patients. [source: [2](#source-2), [3](#source-3), [123](#source-123)]

The reports also show that these impacts are intensified by operational barriers. Damaged infrastructure and continuing seismic instability slowed relief delivery into the hardest-hit areas, and in some locations transportation delays and inadequate infrastructure reduced the speed of aid distribution. [source: [1](#source-1), [2](#source-2), [122](#source-122), [124](#source-124)] One report states that "although resources were abundant, the absence of a unified strategy created redundancies and gaps in aid delivery," showing that coordination problems can limit humanitarian effectiveness even when supplies exist. [source: [124](#source-124)]

Communities are responding through a mix of local action, national coordination, and international humanitarian support. Relief organizations including the Global Relief Network, the Indonesian Red Cross, UNICEF, the Emergency Response Alliance, and the Global Aid Society deployed teams to distribute food, water, clean water, medical aid, and other essential supplies. [source: [1](#source-1), [2](#source-2), [122](#source-122), [123](#source-123)] Volunteers and aid workers established temporary shelters, emergency shelter arrangements, makeshift clinics, and urgent assistance points for displaced people, including in hard-to-access areas such as Medan. [source: [2](#source-2), [122](#source-122), [123](#source-123)] In Denpasar, relief teams coordinated with local authorities to assess damage, set up temporary shelters, and distribute food and water to displaced residents. [source: [123](#source-123)]

Medical response is a central part of the community and humanitarian reaction. The reports note that the Global Relief Network and the World Health Organization sent medical teams to Bali and Lombok for emergency care and preventive health measures, while international specialists supported overwhelmed hospitals with trauma care and infectious disease management. [source: [3](#source-3)] Across affected regions, relief operations also included medical supplies, emergency treatment, makeshift clinics, and disease-mitigation efforts. [source: [1](#source-1), [2](#source-2), [3](#source-3), [121](#source-121), [122](#source-122)]

The response is not limited to immediate relief. Local authorities and the Indonesian government worked with international partners to facilitate aid distribution, welcome outside assistance, and maintain logistical frameworks for relief delivery. [source: [1](#source-1), [2](#source-2), [122](#source-122), [123](#source-123)] Multiple reports say that efforts extend into rebuilding homes, schools, hospitals, infrastructure, and healthcare capacity, alongside psychological support, restoration of essential services, and resilience-building for future disasters. [source: [1](#source-1), [3](#source-3), [121](#source-121), [122](#source-122)] One summary states that response efforts were aimed not only at immediate relief but also at "building resilience and preparing for future seismic events." [source: [1](#source-1)]

In short, the main humanitarian impacts are acute needs for shelter, health care, safe access, and basic services after widespread destruction and displacement, while communities are responding through coordinated emergency aid, volunteer action, medical support, and longer-term reconstruction planning. [source: [1](#source-1), [2](#source-2), [3](#source-3), [121](#source-121), [122](#source-122), [123](#source-123), [124](#source-124)]

## Analysis

### Natural disasters in Indonesia are creating severe humanitarian needs while communities and aid networks respond through emergency relief and longer term recovery

Reports on earthquake-affected areas in Indonesia describe concentrated humanitarian impacts in the hardest-hit communities, including displacement, infrastructure damage, disrupted daily life, and pressure on health services, while also documenting the operational barriers that can slow aid delivery. The material also outlines how local authorities, community volunteers, national institutions, and international organizations are providing emergency shelter, food, water, medical care, and coordination for reconstruction and resilience-building. [source: [1](#source-1), [2](#source-2), [3](#source-3), [121](#source-121), [122](#source-122), [123](#source-123), [124](#source-124)]

#### Theme: Natural disasters in Indonesia are causing acute humanitarian needs in hardest-hit areas while triggering coordinated relief that spans immediate life-saving aid and longer-term recovery.

##### Hardest-hit communities face displacement, damaged infrastructure, disrupted daily life, and strained health systems after the earthquakes.

**Source evidence**: The earthquakes left affected communities in places including Bali, Lombok, Surabaya, Jakarta, Medan, and Denpasar coping with significant destruction, with some regions described as hardest hit and daily life disrupted. [source: [1](#source-1), [2](#source-2), [3](#source-3), [122](#source-122), [123](#source-123)] Infrastructure damage hindered access to some areas, damaged local systems and services, and raised safety concerns for residents and visitors. [source: [1](#source-1), [2](#source-2), [121](#source-121)] Local hospitals were overwhelmed by the influx of patients, while many people were displaced and in need of temporary shelter and urgent assistance. [source: [2](#source-2), [3](#source-3), [123](#source-123)]

**AI commentary**: Taken together, the reports indicate that the main humanitarian impacts are concentrated where physical destruction and access constraints overlap, producing cascading needs in shelter, health care, safety, and continuity of daily life. These conditions suggest that hardest-hit areas require not just more aid, but more sustained and logistically complex support than less-affected locations.

##### Immediate humanitarian response is centered on essential supplies, emergency shelter, clean water, and medical assistance delivered through coordinated local and international operations.

**Source evidence**: Relief organizations including the Global Relief Network, the Indonesian Red Cross, UNICEF, the Emergency Response Alliance, and the Global Aid Society deployed teams to affected areas to distribute food, water, clean water, medical aid, and other essential supplies. [source: [1](#source-1), [2](#source-2), [122](#source-122), [123](#source-123)] Volunteers and aid workers established temporary shelters, emergency shelter arrangements, makeshift clinics, and urgent assistance points for displaced people. [source: [2](#source-2), [122](#source-122), [123](#source-123)] Medical teams supported emergency care, trauma treatment, preventive health measures, and disease-mitigation efforts, while also assisting overwhelmed hospitals. [source: [2](#source-2), [3](#source-3)]

**AI commentary**: The evidence suggests a standard but critical disaster-response sequence: stabilize survival conditions first through supplies, shelter, and health care, especially where displacement and hospital overload are evident. It also implies that operational coordination across organizations is essential because no single actor appears to be covering all humanitarian needs alone.

##### Communities, local authorities, and relief networks are coupling immediate aid with recovery efforts aimed at rebuilding homes, services, and resilience.

**Source evidence**: Local authorities and the Indonesian government worked with international partners to facilitate aid distribution, welcome outside assistance, and maintain logistical frameworks for relief delivery. [source: [1](#source-1), [2](#source-2), [122](#source-122), [123](#source-123)] Multiple reports state that response efforts extend beyond immediate relief to long-term recovery, including rebuilding infrastructure, homes, schools, hospitals, and healthcare capacity. [source: [1](#source-1), [3](#source-3), [121](#source-121), [122](#source-122)] Relief actors also described goals of resilience-building, psychological support, and restoring essential services so affected communities can recover over time. [source: [1](#source-1), [121](#source-121), [122](#source-122)]

**AI commentary**: This pattern indicates that the response is not limited to emergency charity but is evolving into a broader recovery model that links humanitarian relief with reconstruction and preparedness. The inclusion of resilience, psychological support, and infrastructure restoration suggests recognition that disaster impacts are prolonged and that recovery depends on both institutions and community capacity.

#### Theme: Disaster damage and ongoing operational barriers are constraining relief access in Indonesia’s hardest-hit areas despite a large-scale humanitarian response

##### Infrastructure damage and aftershocks are making affected areas harder to reach and slowing relief delivery

**Source evidence**: In Bali, local infrastructure suffered considerable damage after the earthquake, while in Surabaya aftershocks disrupted daily life and the local economy. [source: [1](#source-1)] In Medan, infrastructure damage made access to some areas difficult even as volunteers and aid workers set up temporary shelters and makeshift clinics. [source: [2](#source-2)] Ongoing aftershocks continued to pose a threat and were described as complicating relief efforts and putting additional strain on resources in areas including Surabaya and Lombok. [source: [122](#source-122)]

**AI commentary**: The evidence suggests that physical destruction and continued seismic instability are not only humanitarian concerns in themselves but also direct operational barriers that can delay aid delivery, limit reach into hardest-hit communities, and increase the resources needed to sustain relief operations.

##### Logistical fragmentation and weak coordination are reducing the efficiency of aid operations even where resources are available

**Source evidence**: In Surabaya, the Indonesian Red Cross and the International Relief Fund faced transportation delays and inadequate infrastructure while trying to distribute aid. [source: [124](#source-124)] Officials reported that although resources were abundant, the absence of a unified strategy created redundancies and gaps in aid delivery, and the Indonesian Red Cross said aid was not always reaching intended recipients in a timely manner. [source: [124](#source-124)] The International Relief Fund called for a centralized command and improved communication among stakeholders, while workshops and meetings were being organized to create a more comprehensive distribution plan. [source: [124](#source-124)]

**AI commentary**: This indicates that operational barriers are not limited to damaged roads or facilities; organizational problems can also obstruct relief outcomes, implying that stronger coordination mechanisms may be as important as material supply in determining whether assistance reaches affected populations quickly and equitably.

#### Theme: Coordinated local and international relief networks are delivering immediate aid while organizing longer-term recovery in the hardest-hit disaster areas

##### Relief organizations and local authorities are jointly delivering essential supplies and emergency assistance to hardest-hit communities

**Source evidence**: The Global Relief Network and the Indonesian Red Cross deployed teams to the hardest-hit areas after the Indonesia earthquake and distributed food, water, and medical aid, while the Indonesian government welcomed international assistance for recovery efforts. [source: [1](#source-1)] International organizations including the Global Relief Network and UNICEF worked with local authorities to provide emergency shelter, food, clean water, and medical supplies, and volunteers in Medan set up temporary shelters and makeshift clinics despite access difficulties. [source: [2](#source-2)] In Surabaya and Lombok, the Indonesian Red Cross and Global Aid Society delivered food, clean water, and medical supplies, helped establish temporary shelters, and operated within government logistical frameworks to facilitate aid distribution. [source: [122](#source-122)] In Denpasar, relief teams coordinated with local authorities to assess damage, set up temporary shelters, and distribute food and water to displaced residents. [source: [123](#source-123)]

**AI commentary**: The evidence indicates that frontline humanitarian response depends on multi-level coordination between community actors, local authorities, national governments, and international organizations to move essential supplies quickly into severely affected areas. This suggests that effective disaster relief is not limited to standalone aid delivery but relies on integrated logistics, shared assessments, and partnerships that can sustain operations even when infrastructure damage and aftershocks complicate access.

##### Medical response operations are a central pillar of disaster relief and connect immediate care to health-system recovery

**Source evidence**: The Global Relief Network and the World Health Organization mobilized medical teams to Bali and Lombok to deliver emergency medical care and preventive health measures in the hardest-hit areas after the earthquake. [source: [3](#source-3)] Local hospitals overwhelmed by incoming patients received support from international specialists in trauma care and infectious disease management, and the deployment was intended both to address immediate needs and to support longer-term recovery of healthcare infrastructure. [source: [3](#source-3)] Other relief operations across affected Indonesian regions also included the delivery of medical assistance, medical supplies, and the establishment of makeshift clinics for affected populations. [source: [1](#source-1), [2](#source-2), [121](#source-121), [122](#source-122)]

**AI commentary**: The source material suggests that humanitarian medical action serves a dual function: saving lives immediately after a disaster and preventing secondary crises such as disease outbreaks or prolonged collapse of local health services. This implies that medical deployments are not just an emergency add-on but a bridge between acute response and broader recovery, especially where hospitals are overwhelmed in the hardest-hit zones.

##### Relief efforts are expanding from short-term shelter and basic services toward reconstruction and resilience-building

**Source evidence**: Relief organizations in Bali provided emergency aid, shelter, and medical assistance while also focusing on long-term recovery plans that included reconstruction of homes, schools, and hospitals. [source: [121](#source-121)] Reporting on the broader Indonesian response states that aid efforts were aimed not only at immediate relief but also at building resilience and preparing for future seismic events. [source: [1](#source-1)] In Denpasar, response teams emphasized addressing immediate needs while planning for long-term recovery. [source: [123](#source-123)] In Surabaya and Lombok, responders focused on immediate relief while laying the groundwork for long-term recovery and rebuilding efforts. [source: [122](#source-122)]

**AI commentary**: The evidence points to a recovery model in which emergency shelter, food, water, and medical support are treated as the first phase of a longer reconstruction process rather than as separate tasks. This suggests that communities and institutions are trying to convert relief operations into a platform for rebuilding infrastructure, restoring services, and improving resilience against future disasters.

#### Implications

Taken together, these themes indicate that natural disasters generate both immediate survival needs and longer recovery demands, especially where damaged infrastructure, aftershocks, and coordination gaps obstruct access to the hardest-hit areas. The response patterns show that effective community recovery depends on combining rapid life-saving assistance with stronger logistics, health support, reconstruction planning, and better coordination so aid reaches affected people quickly and recovery can continue beyond the emergency phase. [source: [1](#source-1), [2](#source-2), [3](#source-3), [121](#source-121), [122](#source-122), [123](#source-123), [124](#source-124)]

## Sources

#### Source 1

<details>

##### Text chunk: news_articles_texts.csv_1 (1)

mock_text: **Aftershocks in Indonesia: Local and International Aid Continues**

**Bali, Indonesia – July 16, 2023** – In the wake of the devastating earthquake that struck the Indonesian archipelago, aftershocks continue to rattle the region, with significant tremors felt from Bali to Surabaya. These ongoing seismic activities have heightened the urgency of relief efforts, as both local and international organizations scramble to provide aid and support to the affected communities.

Dr. Emily Chen, a seismologist with extensive experience in earthquake-prone regions, has been closely monitoring the situation. “Aftershocks are a common phenomenon following a major quake,” Dr. Chen explained. “However, the intensity and frequency we’re seeing in Indonesia are exceptional and require sustained attention and resources.”

The Global Relief Network, in conjunction with the Indonesian Red Cross, has been at the forefront of the relief operations. These organizations have deployed teams to the hardest-hit areas, distributing essential supplies such as food, water, and medical aid. The efforts are focused not only on immediate relief but also on building resilience and preparing for potential future seismic events.

In Bali, a popular tourist destination, the local infrastructure has suffered considerable damage, raising concerns about the safety of residents and visitors alike. Meanwhile, in Surabaya, the economic hub of eastern Indonesia, the aftershocks have disrupted daily life, affecting businesses and the local economy.

The Indonesian government has welcomed international assistance, acknowledging the significant contributions from various global entities committed to aiding recovery efforts. The collaboration between local and global organizations exemplifies a unified response to natural disasters, emphasizing the importance of solidarity in times of crisis.

As the people of Indonesia continue to navigate the aftermath of the earthquake, the support from organizations like the Global Relief Network and the Indonesian Red Cross remains critical. With ongoing efforts and increased awareness, the hope is to mitigate the impact of these natural disasters and aid in the recovery and rebuilding of the affected regions.

</details>

[Back to top](#final-report)

#### Source 2

<details>

##### Text chunk: news_articles_texts.csv_2 (1)

mock_text: **International Aid Bolsters Indonesian Relief Efforts**

*July 17, 2023 - Jakarta*

In the wake of the devastating earthquake that struck Indonesia, international organizations have stepped up their efforts to provide much-needed assistance to the affected communities. The earthquake, which primarily impacted the regions of Jakarta and Medan, has left thousands in dire straits, prompting a global response to aid recovery and relief operations.

Dr. Emily Chen, a prominent figure in disaster relief, has been leading the charge for the Global Relief Network. "The situation on the ground is critical," she stated during a press briefing in Jakarta. "Our teams are working around the clock to deliver essential supplies and medical assistance to those in need."

James Park, a field coordinator for UNICEF, echoed Dr. Chen's sentiments. "The international community's support is vital at this juncture. We are collaborating closely with local authorities to ensure that aid is distributed effectively and reaches the most vulnerable populations."

The Global Relief Network and UNICEF are among the many organizations pouring resources into the effort. Both have mobilized their networks to provide emergency shelter, food, clean water, and medical supplies to the affected areas.

In Medan, relief operations are in full swing with volunteers and aid workers setting up temporary shelters and makeshift clinics. The challenges are immense, as infrastructure damage has made access to some areas difficult. However, with increased international aid, there is hope that recovery efforts will gain momentum in the coming weeks.

The Indonesian government has expressed gratitude for the overwhelming support from the international community. Officials are optimistic that with continued assistance, the affected regions will begin to rebuild and recover from the disaster.

As the situation develops, the focus remains on ensuring the safety and well-being of the communities impacted by the earthquake. The collaborative efforts of organizations like the Global Relief Network and UNICEF are crucial in providing a lifeline to those who have lost so much.

</details>

[Back to top](#final-report)

#### Source 3

<details>

##### Text chunk: news_articles_texts.csv_3 (1)

mock_text: **Medical Teams Deployed to Indonesian Islands Post-Quake**

*July 18, 2023*

In the wake of a devastating earthquake that struck the Indonesian islands of Bali and Lombok, international and local efforts are intensifying to provide much-needed relief and medical assistance to the affected populations. The earthquake, which occurred earlier this week, has led to significant destruction across the region, prompting an urgent response from global health organizations.

The Global Relief Network, in collaboration with the World Health Organization, has swiftly mobilized medical teams to the hardest-hit areas. These teams are tasked with delivering essential healthcare services, ranging from emergency medical care to preventive health measures, to mitigate the spread of disease in the aftermath of the disaster.

Dr. Emily Chen, a leading expert in disaster response and a representative of the Global Relief Network, emphasized the critical nature of the situation. "Our primary goal is to stabilize the health conditions in Bali and Lombok and ensure that those affected by the quake receive the care they need as quickly as possible," Dr. Chen stated. "The challenges are immense, but with coordinated efforts, we are determined to make a difference."

Local hospitals, overwhelmed by the influx of patients, are receiving support from these international teams, which include specialists in trauma care and infectious disease management. The deployment aims not only to address immediate medical needs but also to support the longer-term recovery of the healthcare infrastructure in the region.

As the situation continues to evolve, the Global Relief Network and the World Health Organization remain committed to providing ongoing assistance. The efforts are part of a broader strategy to support Indonesia’s recovery and rebuild the resilience of its communities in the face of natural disasters.

</details>

[Back to top](#final-report)

#### Source 121

<details>

##### Text chunk: news_articles_texts.csv_121 (1)

mock_text: **International Teams Bolster Relief in Bali**

*Jakarta, July 17, 2023* — In the wake of a devastating earthquake that shook the Indonesian island of Bali, international relief efforts have ramped up to support the affected communities. The disaster, which struck last week, has left a trail of destruction, prompting a swift response from both local and international organizations.

Dr. Emily Chen, a leading expert in disaster management, has been closely involved in the coordination of relief efforts. She emphasized the importance of rebuilding infrastructure as a crucial step in the recovery process. "Restoring essential services and ensuring that the affected communities have access to basic needs are our top priorities," Dr. Chen stated during a press briefing in Jakarta.

The Global Relief Network, in collaboration with the Indonesian Red Cross, has deployed teams to the hardest-hit areas in Bali. These teams are working tirelessly to provide emergency aid, shelter, and medical assistance to those in need. The organizations have also been focusing on long-term recovery plans, including the reconstruction of homes, schools, and hospitals.

Local authorities have reported that while immediate needs are being met, the road to full recovery will be long. "The resilience of the Balinese people is remarkable," said a spokesperson from the Indonesian Red Cross. "With the continued support from international partners, we are confident that Bali will rebuild stronger than before."

As relief efforts continue, Dr. Emily Chen and her team remain optimistic about the progress being made. "The international community has shown tremendous solidarity," she remarked. "Together, we can help the people of Bali overcome this tragedy and look towards a brighter future."

As the situation unfolds, the focus remains on sustaining the momentum of aid and ensuring that the affected regions receive the necessary support to rebuild and thrive once again.

</details>

[Back to top](#final-report)

#### Source 122

<details>

##### Text chunk: news_articles_texts.csv_122 (1)

mock_text: **Humanitarian Aid Efforts Ramp Up in Indonesia**

*July 18, 2023*

In the wake of recent seismic activity, Indonesia is witnessing a significant increase in humanitarian aid efforts, as both local and international organizations mobilize to support affected communities. The aftermath of the earthquakes has been particularly challenging in regions such as Surabaya and Lombok, where the impact was felt most deeply.

Dr. Emily Chen, a leading disaster response coordinator with the Indonesian Red Cross, has been at the forefront of these efforts. She emphasized the importance of swift action, stating, "Our priority is to ensure that all affected individuals receive the necessary aid without delay. The collaboration between local and international organizations is crucial at this time."

One of the key players in this international collaboration is the Global Aid Society. Anna Martinez, a spokesperson for the organization, highlighted their role in providing essential supplies and expertise. "We are working closely with the Indonesian Red Cross to deliver food, clean water, and medical supplies," she explained. "Our teams are also helping to establish temporary shelters and provide psychological support to those who have lost their homes."

The aftershocks continue to pose a threat, complicating relief efforts and putting additional strain on resources. However, the resilience and dedication of the aid workers are evident as they navigate these challenges. The Indonesian government has also pledged support, ensuring that logistical frameworks are in place to facilitate the distribution of aid.

This concerted effort underscores the importance of international cooperation in times of crisis. As the situation unfolds, the focus remains on providing immediate relief while laying the groundwork for long-term recovery and rebuilding efforts in Surabaya, Lombok, and other affected areas. With the combined efforts of the Indonesian Red Cross, the Global Aid Society, and other partners, there is hope for a swift and effective response to this humanitarian crisis.

</details>

[Back to top](#final-report)

#### Source 123

<details>

##### Text chunk: news_articles_texts.csv_123 (1)

mock_text: **Emergency Response Teams Mobilize in Denpasar Following Seismic Activity**

*Bali, July 19, 2023* — In the wake of recent seismic activity, emergency response teams have been rapidly mobilized in Denpasar, the capital city of Bali. The Global Relief Network, in collaboration with the Emergency Response Alliance, has dispatched personnel to the affected areas to manage the unfolding crisis.

The seismic event, which struck early this morning, has prompted immediate action from local and international agencies. Dr. Emily Chen, a leading figure in disaster management with the Global Relief Network, has arrived on the scene to oversee the coordination of relief efforts. "Our priority is to ensure the safety and well-being of the residents of Denpasar," Dr. Chen stated. "We are working closely with local authorities to assess the damage and provide urgent assistance."

Michael Tan, another key member of the response team from the Emergency Response Alliance, emphasized the importance of swift action. "The situation is evolving, and it is crucial that we address the immediate needs of those affected while planning for long-term recovery," he noted.

Residents of Denpasar have been advised to remain vigilant and follow instructions from local authorities. Relief teams are currently setting up temporary shelters and distributing essential supplies, including food and water, to those displaced by the seismic activity.

The collaboration between the Global Relief Network and the Emergency Response Alliance exemplifies the spirit of international cooperation in times of crisis. Both organizations are committed to providing comprehensive support to help Denpasar recover and rebuild in the aftermath of this event.

As the situation develops, updates will be provided to ensure the community remains informed and prepared. The dedication and resilience of the response teams and local residents alike are crucial in navigating the challenges posed by this natural disaster.

</details>

[Back to top](#final-report)

#### Source 124

<details>

##### Text chunk: news_articles_texts.csv_124 (1)

mock_text: **Aid Distribution Challenges in Surabaya: Calls for Enhanced Coordination**

*July 20, 2023*

In the wake of recent natural disasters, the city of Surabaya has become a focal point for international relief efforts. However, the distribution of aid in this bustling Indonesian city is encountering significant logistical challenges. This situation has prompted urgent calls from key figures and organizations for improved coordination among the entities involved.

Dr. Emily Chen, a renowned expert in disaster management, emphasized the need for streamlined efforts during a recent visit to the affected areas. "Surabaya is facing a complex situation that requires an integrated approach," Dr. Chen stated. "We must ensure that the aid reaches those who need it most, efficiently and effectively."

The Indonesian Red Cross, along with the International Relief Fund, has been at the forefront of the aid distribution efforts. Both organizations have been working tirelessly to overcome obstacles such as transportation delays and inadequate infrastructure. Despite their dedication, the sheer scale of the need in Surabaya and nearby regions like Yogyakarta has highlighted the necessity for better coordination.

Officials have noted that while resources are abundant, the lack of a unified strategy has led to redundancies and gaps in aid delivery. "We are seeing instances where aid is not reaching the intended recipients in a timely manner," noted a spokesperson from the Indonesian Red Cross.

The International Relief Fund has echoed these concerns, calling for a centralized command to oversee operations and improve communication among the various stakeholders. "Our priority is to ensure that the relief efforts are as effective as possible," said the organization's representative.

Efforts are underway to address these challenges. Workshops and meetings between the involved parties are being organized to develop a comprehensive plan that will enhance the efficiency of aid distribution. The hope is that by fostering collaboration, the aid can better serve the communities in need.

As Surabaya and Yogyakarta continue to grapple with the aftermath of the disaster, the focus remains on adapting and improving the delivery of essential support to the affected populations.

</details>

[Back to top](#final-report)



## 7. Condense the answer

Rewrites the full report as a shorter answer. Pass `instructions` to change the style.

In [9]:
condensed_display = show(Markdown(""), display_id=True)
condensed = await qtd.condense_answer(
    query, answer, llm,
    on_token=lambda text: condensed_display.update(Markdown(text)),
)
condensed_display.update(Markdown(condensed))

# Main Humanitarian Impacts and Community Response

Natural disasters in the Indonesia reports caused major displacement, destruction of homes and infrastructure, disruption to daily life and local economies, and heavy pressure on health services. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_3 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

Some of the hardest-hit places mentioned in the reports include Bali, Lombok, Surabaya, Jakarta, Medan, and Denpasar. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_3 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

Damage to roads, buildings, and other infrastructure made some areas difficult to reach and disrupted local systems and services. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_124 (1)]

Aftershocks in places such as Surabaya and Lombok continued to threaten residents and made relief work more difficult. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_122 (1)]

Many people were displaced and needed temporary shelter, food, water, and urgent assistance. [source: news_articles_texts.csv_2 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

Hospitals were overwhelmed by injured and sick people, creating urgent medical needs. [source: news_articles_texts.csv_3 (1)]

## Main Operational Challenges

Relief delivery was slowed by damaged infrastructure, transportation delays, and access problems in affected areas. [source: news_articles_texts.csv_2 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_124 (1)]

The reports also describe coordination problems between aid actors. [source: news_articles_texts.csv_124 (1)]

One report says that resources were available, but the lack of a unified strategy created gaps and overlap in aid delivery. [source: news_articles_texts.csv_124 (1)]

## How Communities and Organizations Are Responding

Communities are responding through local action, government coordination, and support from international and national relief organizations. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

Organizations named in the reports include the Indonesian Red Cross, the Global Relief Network, UNICEF, the World Health Organization, the Emergency Response Alliance, the Global Aid Society, and the International Relief Fund. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_3 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1), news_articles_texts.csv_124 (1)]

These groups and local volunteers distributed essential supplies such as food, water, clean water, medical aid, and medical supplies. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

They also set up temporary shelters, emergency shelter arrangements, and makeshift clinics for displaced people, including in difficult-to-access areas such as Medan and Denpasar. [source: news_articles_texts.csv_2 (1), news_articles_texts.csv_123 (1)]

## Medical Response

Medical response is a central part of the disaster response described in the reports. [source: news_articles_texts.csv_3 (1)]

Medical teams from the Global Relief Network and the World Health Organization were sent to Bali and Lombok to provide emergency care and preventive health services. [source: news_articles_texts.csv_3 (1)]

International specialists also supported overwhelmed hospitals with trauma care and infectious disease management. [source: news_articles_texts.csv_3 (1)]

Other relief operations across affected areas also included medical assistance and makeshift clinics. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1)]

## Longer-Term Recovery

The response is not limited to immediate relief. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

Local authorities and the Indonesian government worked with international partners to support aid delivery and maintain logistical frameworks. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1)]

The reports say recovery efforts also include rebuilding homes, schools, hospitals, infrastructure, and healthcare capacity. [source: news_articles_texts.csv_3 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1)]

Some reports also mention psychological support, restoration of essential services, and resilience-building for future disasters. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1)]

## Summary

In summary, the main humanitarian impacts are destruction, displacement, disrupted services, difficult access, and urgent needs for shelter, health care, food, and water. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_3 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1), news_articles_texts.csv_124 (1)]

Communities are responding through coordinated emergency aid, volunteer support, medical care, temporary shelter, and longer-term rebuilding and recovery efforts. [source: news_articles_texts.csv_1 (1), news_articles_texts.csv_2 (1), news_articles_texts.csv_3 (1), news_articles_texts.csv_121 (1), news_articles_texts.csv_122 (1), news_articles_texts.csv_123 (1), news_articles_texts.csv_124 (1)]